# Watt's the Cost — Subgroup Analysis

**This notebook generates and compares the results of the following models:**
1. Global mean baseline
2. `rate_type_index`-only baseline
3. Categorical-only linear model (`rate_type_index` + `ownership` + `service_type`)
4. Linear model, all features, **no VIF pruning**
5. Linear model, all features, **with VIF pruning** (for comparison — see Section 9 for why this loses accuracy despite "fixing" collinearity)
6. Linear model, all features, **L2-regularized** (an alternative to VIF for taming collinearity without deleting columns)
7. Small neural network, all features
8. Larger neural network, all features

In [1]:
# imports 
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf

# set a seed for reproducibility 
SEED = 0
tf.random.set_seed(SEED)
pd.set_option('display.max_columns', 50)


## 2. Load Data

In [2]:
df = pd.read_parquet('data/cleaned_data.parquet', engine='pyarrow')
print(f'Shape: {df.shape}')

y = df['rate'].copy()
X = df.drop(columns=['rate']).copy()
zip_groups = df['zip'].copy() # keep before zip gets dropped from X, needed for the group split


Shape: (166987, 106)


## 3. Preprocessing

In [3]:
# basic column type fixes 
X = X.drop(columns=['zip'])
X['rate_type_index'] = X['rate_type_index'].astype(str)

utility_cols = ['Utility Res Sales MWh', 'Utility Com Sales MWh', 'Utility Ind Sales MWh']
X[utility_cols] = X[utility_cols].apply(pd.to_numeric, errors='coerce')

# generate a feature column for if row was originall missing data 
for c in utility_cols:
    X[c + '_was_missing'] = X[c].isna().astype(int)


In [4]:
# drop columns with zero variance 
num_cols_all = X.select_dtypes(include=[np.number]).columns
zero_var_cols = [c for c in num_cols_all if X[c].std() == 0 or X[c].nunique() <= 1]
print('Dropping zero-variance columns:', zero_var_cols)
X = X.drop(columns=zero_var_cols)


Dropping zero-variance columns: ['Non-Base Regional Nuclear Pct', 'Non-Base Regional Hydroelectric Pct', 'Non-Base Regional Wind Pct', 'Non-Base Regional Solar Pct', 'Non-Base Regional Geothermal Pct']


In [5]:
# log transform columns that are zero inflated or are skewed
est_cols = [c for c in [
    'est', 'n<5', 'n5_9', 'n10_19', 'n20_49',
    'n50_99', 'n100_249', 'n250_499', 'n500_999', 'n1000',
] if c in X.columns]

zero_inflated_volume_cols = [c for c in utility_cols + ['Utility Total_Generation_MWh'] if c in X.columns]
plain_volume_cols = [c for c in ['total_population', 'households'] if c in X.columns]

for c in zero_inflated_volume_cols:
    X[c + '_is_zero'] = (X[c] == 0).astype(int)

log_candidates = est_cols + zero_inflated_volume_cols + plain_volume_cols
X[log_candidates] = np.log1p(X[log_candidates].clip(lower=0))


In [6]:
# convert race counts to population shares
race_cols = [c for c in X.columns if c in (
    'white_alone', 'black_or_african_american_alone',
    'american_indian_and_alaska_native_alone', 'asian_alone',
    'native_hawaiian_and_other_pacific_islander_alone',
    'some_other_race_alone', 'two_or_more_races:',
    'two_or_more_races:_two_races_including_some_other_race',
    'two_or_more_races:_two_races_excluding_some_other_race,_and_three_or_more_races',
)]
# drop total population and one race reference category
for c in race_cols:
    X[c] = X[c] / X['total_population'].replace(0, np.nan)
X = X.drop(columns=['total_population', 'some_other_race_alone'])

# identify regional and utility energy mix percentage columns
regional_pct_cols = [c for c in X.columns if 'Regional' in c and 'Non-Base' not in c]
nonbase_pct_cols  = [c for c in X.columns if 'Non-Base Regional' in c]
gen_pct_cols      = [c for c in X.columns if c.startswith('Utility ') and c.endswith('_Pct')]

# drop one reference category from each percentage group
reference_cols_to_drop = [c for c in [
    sorted(regional_pct_cols)[0] if regional_pct_cols else None,
    sorted(nonbase_pct_cols)[0] if nonbase_pct_cols else None,
    sorted(gen_pct_cols)[0] if gen_pct_cols else None,
] if c is not None]
X = X.drop(columns=reference_cols_to_drop)

# identify income bracket variables
income_bracket_cols = [c for c in X.columns
                        if c.startswith('households_') and 'median' not in c
                        and 'mean' not in c and 'percent_allocated' not in c]
# drop one income bracket as the reference category
if income_bracket_cols:
    X = X.drop(columns=[sorted(income_bracket_cols)[0]])

print(f'Columns after preprocessing: {X.shape[1]}')


Columns after preprocessing: 100


## 4. Optional: VIF Pruning

Method for removing correlated variables (affects linear regression model interpretability).

## 5. Train / Val / Test Split — Grouped by ZIP

Prevents the same ZIP from appearing in both train and test, which would otherwise let the model partly memorize ZIP-level features rather than generalize from them.

In [7]:
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(gss1.split(X, y, groups=zip_groups))
X_train, X_temp = X.iloc[train_idx].copy(), X.iloc[temp_idx].copy()
y_train, y_temp = y.iloc[train_idx].copy(), y.iloc[temp_idx].copy()
zip_temp = zip_groups.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(gss2.split(X_temp, y_temp, groups=zip_temp))
X_val, X_test = X_temp.iloc[val_idx].copy(), X_temp.iloc[test_idx].copy()
y_val, y_test = y_temp.iloc[val_idx].copy(), y_temp.iloc[test_idx].copy()

train_zips = set(zip_groups.iloc[train_idx])
val_zips   = set(zip_groups.iloc[temp_idx].iloc[val_idx])
test_zips  = set(zip_groups.iloc[temp_idx].iloc[test_idx])
assert not (train_zips & val_zips) and not (train_zips & test_zips) and not (val_zips & test_zips)
print('No ZIP overlap between splits - confirmed.')

utility_cols_present = [c for c in utility_cols if c in X_train.columns]
utility_medians = X_train[utility_cols_present].median()
for split in (X_train, X_val, X_test):
    split[utility_cols_present] = split[utility_cols_present].fillna(utility_medians)

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')


No ZIP overlap between splits - confirmed.
Train: (116731, 100)  Val: (25070, 100)  Test: (25186, 100)


## 6. Train + Evaluate Helper Functions

## 7. Model Builders

In [8]:
def build_linear(num_features, learning_rate):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    model = tf.keras.Sequential([tf.keras.layers.Dense(1, input_shape=(num_features,))])
    model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
                   loss='mse', metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')])
    return model


def build_linear_l2(num_features, learning_rate, l2=0.01):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    model = tf.keras.Sequential([tf.keras.layers.Dense(
        1, input_shape=(num_features,), kernel_regularizer=tf.keras.regularizers.l2(l2))])
    model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
                   loss='mse', metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')])
    return model


def build_nn(num_features, learning_rate, hidden_units=(64, 32), clipnorm=1.0):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    model = tf.keras.Sequential([tf.keras.layers.Input(shape=(num_features,))])
    for units in hidden_units:
        model.add(tf.keras.layers.Dense(units, activation='relu'))
    model.add(tf.keras.layers.Dense(1))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='mse', metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')])
    return model


In [9]:
all_cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
all_num_cols = [c for c in X.columns if c not in all_cat_cols]

final_preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), all_cat_cols),
    ('num', StandardScaler(), all_num_cols),
])

Xtr_final = final_preprocessor.fit_transform(X_train[all_cat_cols + all_num_cols])
Xv_final = final_preprocessor.transform(X_val[all_cat_cols + all_num_cols])
Xte_final = final_preprocessor.transform(X_test[all_cat_cols + all_num_cols])

In [10]:
l2_model = build_linear_l2(Xtr_final.shape[1], 0.01)
l2_model.fit(Xtr_final, y_train, epochs=100, batch_size=2048, verbose=0,
                 callbacks=[tf.keras.callbacks.EarlyStopping(monitor='loss', patience=8, restore_best_weights=True)])

2026-08-05 16:18:06.495817: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-08-05 16:18:06.496431: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-08-05 16:18:06.496459: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-08-05 16:18:06.496899: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-08-05 16:18:06.497881: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-08-05 16:18:07.229145: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-08-05 16:18:07.247835: E te

In [11]:
final_nn = build_nn(
    Xtr_final.shape[1],
    learning_rate=0.001,
    hidden_units=(32, 16)
)

# Train
history = final_nn.fit(
    Xtr_final,
    y_train,
    validation_data=(Xv_final, y_val),
    epochs=200,
    batch_size=2048,
    verbose=0,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=8,
            restore_best_weights=True
        )
    ]
)

In [12]:
# Predict using the FINAL L2 model
test_preds = l2_model.predict(Xte_final, verbose=0).flatten()

# Store predictions with true labels and subgroup labels
subgroup_df = pd.DataFrame({
    "rate_type_index": X_test["rate_type_index"].values,
    "actual": y_test.values,
    "predicted": test_preds
})

rate_map = {
    "0": "Commercial",
    "1": "Industrial",
    "2": "Residential"
}
subgroup_df["rate_type"] = subgroup_df["rate_type_index"].map(rate_map)

subgroup_results = []

for rate_type, group in subgroup_df.groupby("rate_type"):
    subgroup_results.append({
        "Rate Type": rate_type,
        "Samples": len(group),
        "RMSE": np.sqrt(mean_squared_error(group["actual"], group["predicted"])),
        "MAE": mean_absolute_error(group["actual"], group["predicted"])
    })

subgroup_results = pd.DataFrame(subgroup_results)

print(subgroup_results)

     Rate Type  Samples      RMSE       MAE
0   Commercial     9071  0.029356  0.020407
1   Industrial     8928  0.030810  0.022164
2  Residential     7187  0.035630  0.026052


In [13]:
test_preds = final_nn.predict(Xte_final, verbose=0).flatten()

subgroup_df = pd.DataFrame({
    "rate_type_index": X_test["rate_type_index"].values,
    "actual": y_test.values,
    "predicted": test_preds
})

rate_map = {
    "0": "Commercial",
    "1": "Industrial",
    "2": "Residential"
}
subgroup_df["rate_type"] = subgroup_df["rate_type_index"].map(rate_map)

subgroup_results = []

for rate_type, group in subgroup_df.groupby("rate_type"):
    subgroup_results.append({
        "Rate Type": rate_type,
        "Samples": len(group),
        "RMSE": np.sqrt(mean_squared_error(group["actual"], group["predicted"])),
        "MAE": mean_absolute_error(group["actual"], group["predicted"])
    })

subgroup_results = pd.DataFrame(subgroup_results)

print(subgroup_results)

     Rate Type  Samples      RMSE       MAE
0   Commercial     9071  0.031826  0.021680
1   Industrial     8928  0.034120  0.023495
2  Residential     7187  0.037978  0.027263
